In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_09/data'
os.listdir(data_dir)

['campaigns.csv', 'sessions.csv', 'orders.csv', 'ad_spend.csv']

In [2]:
import pandas as pd
orders = pd.read_csv(os.path.join(data_dir, 'orders.csv'))
sessions = pd.read_csv(os.path.join(data_dir, 'sessions.csv'))
print("Orders columns:", orders.columns)
print("Sessions columns:", sessions.columns)
print(orders.head())
print(sessions.head())

Orders columns: Index(['order_id', 'session_id', 'order_value', 'items'], dtype='str')
Sessions columns: Index(['session_id', 'session_date', 'user_id', 'campaign_id', 'pages_viewed',
       'session_seconds', 'device'],
      dtype='str')
  order_id session_id  order_value  items
0    O0001    SSN0002           71      1
1    O0002    SSN0005          107      1
2    O0003    SSN0007          134      4
3    O0004    SSN0015           93      4
4    O0005    SSN0020          156      3
  session_id session_date user_id campaign_id  pages_viewed  session_seconds  \
0    SSN0001   2024-03-01     U01         C04             3               48   
1    SSN0002   2024-03-01     U03         C08             5               54   
2    SSN0003   2024-03-01     U04         C02             1               57   
3    SSN0004   2024-03-01     U06         C06             3               63   
4    SSN0005   2024-03-01     U07         C08             4               66   

    device  
0   tablet  
1

In [3]:
campaigns = pd.read_csv(os.path.join(data_dir, 'campaigns.csv'))
print("Campaigns columns:", campaigns.columns)
print(campaigns.head())

Campaigns columns: Index(['campaign_id', 'channel', 'region'], dtype='str')
  campaign_id  channel region
0         C01   search     US
1         C02   social     US
2         C03    email     EU
3         C04  display     EU
4         C05   search   APAC


In [4]:
# Merge orders with sessions to find the user and date of each order
order_sessions = orders.merge(sessions, on='session_id', how='inner')

# Find the first purchase date for each user
first_purchase = order_sessions.groupby('user_id')['session_date'].min().reset_index()
first_purchase.rename(columns={'session_date': 'first_purchase_date'}, inplace=True)

# Merge sessions with campaigns to get the channel
sessions_with_channel = sessions.merge(campaigns, on='campaign_id', how='left')

# Merge first purchase date back to sessions
user_sessions = sessions_with_channel.merge(first_purchase, on='user_id', how='inner')

# Filter sessions to those on or before the first purchase date
lead_up_sessions = user_sessions[user_sessions['session_date'] <= user_sessions['first_purchase_date']]

# Count unique channels per user in the lead-up
channel_counts = lead_up_sessions.groupby('user_id')['channel'].nunique()

# Count how many users had > 1 channel
users_with_multiple_channels = (channel_counts > 1).sum()
print("Users with > 1 channel:", users_with_multiple_channels)

Users with > 1 channel: 13


In [5]:
# Let's double check the logic.
# 1. Find users who eventually bought.
# 2. Find their first purchase date.
# 3. Look at all their sessions up to and including the first purchase date.
# 4. Count the number of unique channels in those sessions.
# 5. Count how many users have > 1 unique channel.

# Let's verify the first purchase date logic.
# order_sessions has all orders and their corresponding session info.
# first_purchase has the minimum session_date for each user who made an order.
# user_sessions has all sessions for all users, merged with their first purchase date (inner join, so only users who bought are kept).
# lead_up_sessions filters to sessions where session_date <= first_purchase_date.
# channel_counts counts unique channels per user in lead_up_sessions.
# users_with_multiple_channels counts users where channel_counts > 1.

# Let's check if there are any missing values or issues.
print(order_sessions.head())
print(first_purchase.head())
print(lead_up_sessions.head())
print(channel_counts.head())
print(users_with_multiple_channels)

  order_id session_id  order_value  items session_date user_id campaign_id  \
0    O0001    SSN0002           71      1   2024-03-01     U03         C08   
1    O0002    SSN0005          107      1   2024-03-01     U07         C08   
2    O0003    SSN0007          134      4   2024-03-01     U10         C06   
3    O0004    SSN0015           93      4   2024-03-02     U05         C05   
4    O0005    SSN0020          156      3   2024-03-02     U12         C03   

   pages_viewed  session_seconds   device  
0             5               54  desktop  
1             4               66   tablet  
2             2               75   tablet  
3             3               65  desktop  
4             5               86   tablet  
  user_id first_purchase_date
0     U01          2024-03-03
1     U02          2024-03-05
2     U03          2024-03-01
3     U04          2024-03-06
4     U05          2024-03-02
  session_id session_date user_id campaign_id  pages_viewed  session_seconds  \
0    SS

In [6]:
print(sessions.head(10))
print(sessions.tail(10))

  session_id session_date user_id campaign_id  pages_viewed  session_seconds  \
0    SSN0001   2024-03-01     U01         C04             3               48   
1    SSN0002   2024-03-01     U03         C08             5               54   
2    SSN0003   2024-03-01     U04         C02             1               57   
3    SSN0004   2024-03-01     U06         C06             3               63   
4    SSN0005   2024-03-01     U07         C08             4               66   
5    SSN0006   2024-03-01     U09         C04             1               72   
6    SSN0007   2024-03-01     U10         C06             2               75   
7    SSN0008   2024-03-01     U12         C02             4               81   
8    SSN0009   2024-03-01     U13         C04             5               84   
9    SSN0010   2024-03-01     U15         C08             2               90   

    device  
0   tablet  
1  desktop  
2   tablet  
3  desktop  
4   tablet  
5  desktop  
6   tablet  
7  desktop  
8 

In [7]:
# Let's check if there are multiple sessions on the same day for a user.
# If so, we might need to use session_id or a timestamp if available.
# The data only has session_date.
# "in the lead-up to their first purchase"
# If a user has multiple sessions on the day of their first purchase, do they all count as "lead-up"?
# Usually, yes, or we just use all sessions up to that date.
# Let's check if there's a time component.
print(sessions['session_date'].unique())

<StringArray>
['2024-03-01', '2024-03-02', '2024-03-03', '2024-03-04', '2024-03-05',
 '2024-03-06', '2024-03-07', '2024-03-08', '2024-03-09', '2024-03-10',
 '2024-03-11', '2024-03-12', '2024-03-13', '2024-03-14', '2024-03-15',
 '2024-03-16', '2024-03-17', '2024-03-18', '2024-03-19', '2024-03-20',
 '2024-03-21', '2024-03-22', '2024-03-23', '2024-03-24', '2024-03-25',
 '2024-03-26', '2024-03-27', '2024-03-28', '2024-03-29', '2024-03-30',
 '2024-03-31']
Length: 31, dtype: str


In [8]:
# Let's review the question: "Among users who eventually bought, how many had used more than one channel in the lead-up to their first purchase?"
# "lead-up to their first purchase" usually includes the session where the purchase happened.
# Let's check if there are any users who bought multiple times on their first purchase date.
# Let's check if there are any users who had multiple sessions on their first purchase date.
# Let's check if there are any users who had multiple channels on their first purchase date.

# Let's re-read the question carefully.
# "Among users who eventually bought, how many had used more than one channel in the lead-up to their first purchase?"
# This means we need to find the first purchase session for each user.
# Then we need to find all sessions for that user that occurred *before or during* that first purchase session.
# Since we only have session_date, we can assume sessions on the same date as the first purchase are part of the lead-up.
# Or, we can assume the sessions are ordered by session_id?
# Let's check if session_id is ordered.
print(sessions['session_id'].head(20))
print(sessions['session_id'].tail(20))

0     SSN0001
1     SSN0002
2     SSN0003
3     SSN0004
4     SSN0005
5     SSN0006
6     SSN0007
7     SSN0008
8     SSN0009
9     SSN0010
10    SSN0011
11    SSN0012
12    SSN0013
13    SSN0014
14    SSN0015
15    SSN0016
16    SSN0017
17    SSN0018
18    SSN0019
19    SSN0020
Name: session_id, dtype: str
352    SSN0353
353    SSN0354
354    SSN0355
355    SSN0356
356    SSN0357
357    SSN0358
358    SSN0359
359    SSN0360
360    SSN0361
361    SSN0362
362    SSN0363
363    SSN0364
364    SSN0365
365    SSN0366
366    SSN0367
367    SSN0368
368    SSN0369
369    SSN0370
370    SSN0371
371    SSN0372
Name: session_id, dtype: str


In [9]:
# Let's check if session_id is ordered by time.
# If so, we can use session_id to determine the order of sessions.
# Let's extract the number from session_id and sort by it.
sessions['session_num'] = sessions['session_id'].str.extract('(\d+)').astype(int)
print(sessions.sort_values('session_num').head())
print(sessions.sort_values('session_num').tail())

  session_id session_date user_id campaign_id  pages_viewed  session_seconds  \
0    SSN0001   2024-03-01     U01         C04             3               48   
1    SSN0002   2024-03-01     U03         C08             5               54   
2    SSN0003   2024-03-01     U04         C02             1               57   
3    SSN0004   2024-03-01     U06         C06             3               63   
4    SSN0005   2024-03-01     U07         C08             4               66   

    device  session_num  
0   tablet            1  
1  desktop            2  
2   tablet            3  
3  desktop            4  
4   tablet            5  
    session_id session_date user_id campaign_id  pages_viewed  \
367    SSN0368   2024-03-31     U12         C08             4   
368    SSN0369   2024-03-31     U13         C02             5   
369    SSN0370   2024-03-31     U15         C06             2   
370    SSN0371   2024-03-31     U16         C08             3   
371    SSN0372   2024-03-31     U18   

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_408172/1992902906.py:4: SyntaxWarning: invalid escape sequence '\d'
  sessions['session_num'] = sessions['session_id'].str.extract('(\d+)').astype(int)


In [10]:
# Let's use session_num to find the first purchase session for each user.
order_sessions = orders.merge(sessions, on='session_id', how='inner')
first_purchase_session = order_sessions.groupby('user_id')['session_num'].min().reset_index()
first_purchase_session.rename(columns={'session_num': 'first_purchase_session_num'}, inplace=True)

# Merge sessions with campaigns to get the channel
sessions_with_channel = sessions.merge(campaigns, on='campaign_id', how='left')

# Merge first purchase session back to sessions
user_sessions = sessions_with_channel.merge(first_purchase_session, on='user_id', how='inner')

# Filter sessions to those on or before the first purchase session
lead_up_sessions = user_sessions[user_sessions['session_num'] <= user_sessions['first_purchase_session_num']]

# Count unique channels per user in the lead-up
channel_counts = lead_up_sessions.groupby('user_id')['channel'].nunique()

# Count how many users had > 1 channel
users_with_multiple_channels = (channel_counts > 1).sum()
print("Users with > 1 channel:", users_with_multiple_channels)

Users with > 1 channel: 13
